# 12 · Extensión híbrida del recomendador: Pareto + KNN

En los notebooks anteriores se ha construido y evaluado un recomendador cuyo núcleo es el equilibrio entre **precio de la vivienda** y **tiempo de desplazamiento**. Para ello se han utilizado la frontera de Pareto, perfiles de usuario y distintas comprobaciones de sensibilidad y robustez.

Este notebook incorpora **K-Nearest Neighbors (KNN)** como una extensión del sistema. Su función no es sustituir al análisis multiobjetivo ni predecir precios, sino añadir una capa de **similitud entre barrios**.

La lógica es la siguiente:

1. Para cada destino y perfil se selecciona una recomendación de referencia dentro de la frontera de Pareto.
2. KNN busca barrios similares a esa recomendación.
3. Se compara un **KNN global**, que puede recuperar cualquier barrio disponible, con un enfoque **Pareto + KNN**, que limita la búsqueda a alternativas no dominadas.
4. Se analiza el compromiso entre **similitud** y **eficiencia multiobjetivo**.
5. Se estudia la sensibilidad al número de vecinos y el grado de solapamiento entre perfiles.

De esta forma, el aprendizaje automático aporta una función concreta al recomendador sin desplazar el problema principal del TFG: encontrar zonas que ofrezcan un buen compromiso entre coste de vivienda y accesibilidad.

## 12.1 Pregunta de análisis

La pregunta principal de este notebook es:

> **¿Puede KNN aportar alternativas similares y útiles a la recomendación principal sin empeorar el equilibrio entre precio y tiempo de desplazamiento?**

Se comparan dos estrategias:

- **KNN global:** busca vecinos similares entre todos los barrios disponibles para un destino.
- **Pareto + KNN:** busca vecinos únicamente entre los barrios que pertenecen a la frontera de Pareto.

La segunda estrategia representa la extensión híbrida propuesta.

## 12.2 Hipótesis

Se plantean dos hipótesis:

- **H1.** KNN permite recuperar barrios próximos a la recomendación principal en términos de precio, accesibilidad y características residenciales.
- **H2.** Restringir KNN a la frontera de Pareto aumenta la proporción de alternativas eficientes sin provocar una pérdida excesiva de similitud.

KNN se utiliza aquí como método de **recuperación por similitud**, no como modelo predictivo. Por este motivo no se emplean métricas como *accuracy*, MAE o R². La evaluación se centra en la calidad, coherencia y estabilidad de las alternativas recuperadas.

## 12.3 Librerías y configuración

Se utilizan `pandas` y `numpy` para el tratamiento de datos, `matplotlib` para las visualizaciones y `scikit-learn` para estandarizar las variables y aplicar KNN.

Se fija una semilla por reproducibilidad, aunque `NearestNeighbors` con distancia euclídea es determinista para un mismo conjunto de datos.

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Configuración de visualización para facilitar la inspección.
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

## 12.4 Localización del proyecto y de los archivos

La entrada principal es `pareto_by_destination.csv`, generado en la parte multiobjetivo del proyecto.

El notebook localiza automáticamente la raíz del proyecto buscando la carpeta `data`. Esto evita depender de una ruta absoluta y permite ejecutar el notebook tanto desde la carpeta principal como desde `notebooks/`.

In [ ]:
def find_project_root():
    # Partimos del directorio actual y subimos por sus carpetas padre.
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la carpeta 'data'. "
        "Ejecuta el notebook desde el proyecto TFG_Recomendador_Viviendas."
    )


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RESULTS_DIR / "pareto_by_destination.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_PATH)

## 12.5 Carga del dataset

Antes de continuar se comprueba que el archivo exista. A continuación se muestran sus dimensiones y las primeras filas para verificar que la estructura corresponde con la salida esperada de los notebooks anteriores.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No existe el archivo {INPUT_PATH}. "
        "Comprueba que se han ejecutado correctamente los notebooks anteriores."
    )

data = pd.read_csv(INPUT_PATH)

print(f"Filas: {len(data):,}")
print(f"Columnas: {data.shape[1]:,}")
display(data.head())

### Análisis inicial

En este punto interesa comprobar especialmente que exista una fila por combinación barrio–destino y que estén disponibles los dos objetivos principales: precio por metro cuadrado y tiempo diario de desplazamiento.

Los casos sin información suficiente no deben inventarse ni imputarse a nivel de objetivo. KNN se aplicará únicamente sobre alternativas que dispongan de precio y tiempo válidos.

## 12.6 Identificación robusta de columnas

Para que el notebook tolere pequeños cambios de nomenclatura, se buscan las columnas principales entre varios nombres posibles.

Las variables obligatorias son:

- identificador del barrio;
- destino;
- precio por metro cuadrado utilizado por el recomendador;
- tiempo diario de desplazamiento.

Las características residenciales adicionales se incorporarán después solo cuando existan realmente.

In [ ]:
def first_existing_column(df, candidates, required=False, label=None):
    # Devuelve la primera columna disponible entre los nombres candidatos.
    for column in candidates:
        if column in df.columns:
            return column

    if required:
        raise KeyError(
            f"No se ha encontrado la columna necesaria para {label or candidates}. "
            f"Candidatas: {candidates}"
        )

    return None


ZONE_ID_COL = first_existing_column(
    data,
    ["zone_key", "neighborhood_id", "zone_id"],
    required=True,
    label="identificador del barrio",
)

ZONE_NAME_COL = first_existing_column(
    data,
    ["zone_name", "neighborhood_name", "barrio", "name"],
)

DEST_COL = first_existing_column(
    data,
    ["destination_name", "destination"],
    required=True,
    label="destino",
)

PRICE_COL = first_existing_column(
    data,
    ["price_m2_for_recommender", "price_m2", "registered_price_m2_total_2025"],
    required=True,
    label="precio por metro cuadrado",
)

TIME_COL = first_existing_column(
    data,
    ["commute_daily_minutes", "commute_minutes", "travel_time_minutes"],
    required=True,
    label="tiempo de desplazamiento",
)

print("Identificador de barrio:", ZONE_ID_COL)
print("Nombre de barrio:", ZONE_NAME_COL)
print("Destino:", DEST_COL)
print("Precio:", PRICE_COL)
print("Tiempo:", TIME_COL)

## 12.7 Validación y preparación inicial

No se imputan los dos objetivos principales. Una alternativa sin precio o sin tiempo no puede compararse correctamente en el plano multiobjetivo y, por tanto, se excluye de esta extensión.

También se comprueba que los precios sean positivos, que los tiempos no sean negativos y que no existan duplicados barrio–destino.

In [ ]:
work = data.copy()

# Convertimos explícitamente los dos objetivos a formato numérico.
work[PRICE_COL] = pd.to_numeric(work[PRICE_COL], errors="coerce")
work[TIME_COL] = pd.to_numeric(work[TIME_COL], errors="coerce")

initial_rows = len(work)

# Solo conservamos filas comparables en precio y tiempo.
work = work.dropna(
    subset=[ZONE_ID_COL, DEST_COL, PRICE_COL, TIME_COL]
).copy()

print(f"Filas iniciales: {initial_rows:,}")
print(f"Filas válidas para KNN: {len(work):,}")
print(f"Filas descartadas: {initial_rows - len(work):,}")
print(f"Destinos: {work[DEST_COL].nunique():,}")
print(f"Barrios únicos: {work[ZONE_ID_COL].nunique():,}")

if work.empty:
    raise ValueError("No quedan filas válidas después de la validación.")

if (work[PRICE_COL] <= 0).any():
    raise ValueError("Se han encontrado precios no positivos.")

if (work[TIME_COL] < 0).any():
    raise ValueError("Se han encontrado tiempos de desplazamiento negativos.")

duplicates = work.duplicated(
    subset=[ZONE_ID_COL, DEST_COL]
).sum()

print(f"Duplicados barrio-destino: {duplicates:,}")

if duplicates > 0:
    raise ValueError(
        "Existen combinaciones barrio-destino duplicadas. "
        "Conviene revisar el dataset antes de aplicar KNN."
    )

In [ ]:
coverage_pct = 100 * len(work) / initial_rows if initial_rows else np.nan

display(
    Markdown(
        f"""
### Resultado de la validación

De las **{initial_rows:,} filas** disponibles inicialmente, **{len(work):,}** pueden compararse
simultáneamente en precio y tiempo, lo que representa una cobertura del **{coverage_pct:.2f} %**.

El análisis KNN se realizará sobre **{work[ZONE_ID_COL].nunique():,} barrios** y
**{work[DEST_COL].nunique():,} destinos**. Los casos sin información completa se mantienen
fuera de esta extensión para no introducir valores de precio o transporte que no han sido observados.
"""
    )
)

## 12.8 Recalcular la frontera de Pareto

Aunque la entrada procede del análisis multiobjetivo, se recalcula la condición de Pareto para que este notebook sea autocontenido.

Un barrio está **dominado** si existe otra alternativa que:

- no sea más cara;
- no tenga un tiempo de desplazamiento mayor;
- y mejore estrictamente al menos uno de esos dos objetivos.

Los dos objetivos se minimizan.

In [ ]:
def pareto_mask_minimize(df, price_col, time_col):
    # Matriz con los dos objetivos a minimizar.
    values = df[[price_col, time_col]].to_numpy(dtype=float)

    # Inicialmente suponemos que todas las alternativas son eficientes.
    efficient = np.ones(len(values), dtype=bool)

    for i, point in enumerate(values):
        # Otra fila domina al punto si no empeora ningún objetivo
        # y mejora estrictamente al menos uno.
        dominates_point = (
            (values[:, 0] <= point[0])
            & (values[:, 1] <= point[1])
            & (
                (values[:, 0] < point[0])
                | (values[:, 1] < point[1])
            )
        )

        dominates_point[i] = False
        efficient[i] = not dominates_point.any()

    return efficient


work["is_pareto_knn12"] = False

for destination, indices in work.groupby(DEST_COL).groups.items():
    group = work.loc[indices]

    work.loc[indices, "is_pareto_knn12"] = pareto_mask_minimize(
        group,
        PRICE_COL,
        TIME_COL,
    )

pareto_summary = (
    work.groupby(DEST_COL)
    .agg(
        alternatives=(ZONE_ID_COL, "size"),
        pareto_alternatives=("is_pareto_knn12", "sum"),
    )
    .reset_index()
)

pareto_summary["pareto_percentage"] = (
    100
    * pareto_summary["pareto_alternatives"]
    / pareto_summary["alternatives"]
)

display(pareto_summary)

In [ ]:
mean_pareto_pct = pareto_summary["pareto_percentage"].mean()
min_pareto_row = pareto_summary.loc[
    pareto_summary["pareto_percentage"].idxmin()
]
max_pareto_row = pareto_summary.loc[
    pareto_summary["pareto_percentage"].idxmax()
]

display(
    Markdown(
        f"""
### Análisis de la frontera utilizada por KNN

De media, el **{mean_pareto_pct:.2f} %** de las alternativas de cada destino pertenece a su
frontera de Pareto.

El frente más reducido corresponde a **{min_pareto_row[DEST_COL]}**, con
**{min_pareto_row['pareto_percentage']:.2f} %** de alternativas no dominadas, mientras que
el más amplio corresponde a **{max_pareto_row[DEST_COL]}**, con
**{max_pareto_row['pareto_percentage']:.2f} %**.

Cuanto más pequeño sea el frente, más restrictiva será la búsqueda Pareto + KNN y mayor
puede ser el coste en similitud respecto al KNN global.
"""
    )
)

## 12.9 Perfiles de usuario

Se mantienen tres perfiles sencillos e interpretables:

- **Estudiante:** prioriza claramente el precio.
- **Equilibrado:** asigna la misma importancia a precio y tiempo.
- **Profesional:** prioriza el tiempo de desplazamiento.

Los pesos se usan únicamente para seleccionar el barrio de referencia. Después, KNN se ocupa de buscar alternativas similares.

In [ ]:
PROFILES = {
    "estudiante": {
        "weight_price": 0.80,
        "weight_time": 0.20,
    },
    "equilibrado": {
        "weight_price": 0.50,
        "weight_time": 0.50,
    },
    "profesional": {
        "weight_price": 0.25,
        "weight_time": 0.75,
    },
}

profiles_df = pd.DataFrame(PROFILES).T.reset_index(names="profile")

assert np.allclose(
    profiles_df["weight_price"] + profiles_df["weight_time"],
    1.0,
), "Los pesos de todos los perfiles deben sumar 1."

display(profiles_df)

## 12.10 Selección de la recomendación de referencia

Para cada destino se normalizan precio y tiempo mediante *min-max scaling*. Después se calcula una puntuación ponderada según el perfil.

La recomendación de referencia se selecciona **únicamente entre barrios Pareto eficientes**. La puntuación se minimiza: un valor menor representa una combinación más favorable para el perfil.

In [ ]:
def minmax(series):
    # Normalización min-max dentro de cada destino.
    series = pd.to_numeric(series, errors="coerce")

    min_value = series.min()
    max_value = series.max()

    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)

    # Si todos los valores son iguales, la variable no discrimina.
    if np.isclose(min_value, max_value):
        return pd.Series(0.0, index=series.index)

    return (series - min_value) / (max_value - min_value)


anchor_rows = []

for destination, group in work.groupby(DEST_COL):
    group = group.copy()

    group["price_norm_12"] = minmax(group[PRICE_COL])
    group["time_norm_12"] = minmax(group[TIME_COL])

    pareto_group = group[
        group["is_pareto_knn12"]
    ].copy()

    if pareto_group.empty:
        continue

    for profile_name, profile in PROFILES.items():
        pareto_group["profile_score_12"] = (
            profile["weight_price"] * pareto_group["price_norm_12"]
            + profile["weight_time"] * pareto_group["time_norm_12"]
        )

        # Precio, tiempo e identificador se usan como desempate determinista.
        anchor = pareto_group.sort_values(
            ["profile_score_12", PRICE_COL, TIME_COL, ZONE_ID_COL]
        ).iloc[0]

        row = {
            DEST_COL: destination,
            "profile": profile_name,
            "anchor_zone_id": anchor[ZONE_ID_COL],
            "anchor_price_m2": anchor[PRICE_COL],
            "anchor_commute_minutes": anchor[TIME_COL],
            "anchor_score": anchor["profile_score_12"],
            "anchor_is_pareto": bool(anchor["is_pareto_knn12"]),
        }

        if ZONE_NAME_COL is not None:
            row["anchor_zone_name"] = anchor[ZONE_NAME_COL]

        anchor_rows.append(row)

anchors = pd.DataFrame(anchor_rows)

assert not anchors.empty
assert anchors["anchor_is_pareto"].all()

display(anchors.sort_values([DEST_COL, "profile"]))

In [ ]:
anchor_diversity = (
    anchors.groupby(DEST_COL)["anchor_zone_id"]
    .nunique()
    .reset_index(name="different_anchors")
)

display(anchor_diversity)

mean_anchor_diversity = anchor_diversity["different_anchors"].mean()
same_anchor_destinations = (
    anchor_diversity["different_anchors"] == 1
).sum()

display(
    Markdown(
        f"""
### Análisis de las recomendaciones de referencia

El número medio de barrios de referencia distintos por destino es
**{mean_anchor_diversity:.2f}**.

En **{same_anchor_destinations} de {len(anchor_diversity)} destinos**, los tres perfiles
seleccionan el mismo barrio. Cuando aparecen varias referencias, modificar la importancia
relativa del precio y del tiempo sí cambia la solución preferida.

Este comportamiento servirá después para comprobar si las listas KNN también conservan
diferencias entre perfiles.
"""
    )
)

## 12.11 Variables utilizadas por KNN

KNN representa cada barrio mediante un vector de características.

Se incluyen siempre:

- precio por metro cuadrado;
- tiempo diario de desplazamiento.

Cuando están disponibles se incorporan además variables residenciales interpretables como superficie, habitaciones, baños, ascensor o exterior.

Estas variables añaden contexto: dos barrios pueden tener precio y tiempo similares, pero ofrecer un parque residencial diferente.

In [ ]:
OPTIONAL_FEATURE_GROUPS = {
    "superficie": [
        "size_mean",
        "size_median",
        "sale_size",
        "metros_mean",
        "metros_median",
    ],
    "habitaciones": [
        "rooms_mean",
        "sale_rooms",
        "habitaciones_mean",
    ],
    "banos": [
        "baths_mean",
        "sale_baths",
        "banos_mean",
    ],
    "ascensor": [
        "elevator_ratio",
        "ascensor_ratio",
    ],
    "exterior": [
        "exterior_ratio",
    ],
}

optional_features = []

for label, candidates in OPTIONAL_FEATURE_GROUPS.items():
    column = first_existing_column(
        work,
        candidates,
        required=False,
    )

    if column is not None and column not in optional_features:
        optional_features.append(column)

KNN_FEATURES = [PRICE_COL, TIME_COL] + optional_features

# Convertimos a numérico todas las variables que entren en la distancia.
for column in KNN_FEATURES:
    work[column] = pd.to_numeric(
        work[column],
        errors="coerce",
    )

feature_info = pd.DataFrame(
    {
        "feature": KNN_FEATURES,
        "missing_pct": [
            100 * work[column].isna().mean()
            for column in KNN_FEATURES
        ],
        "n_unique": [
            work[column].nunique(dropna=True)
            for column in KNN_FEATURES
        ],
    }
)

display(feature_info)
print("Variables utilizadas por KNN:", KNN_FEATURES)

In [ ]:
optional_text = (
    ", ".join(optional_features)
    if optional_features
    else "ninguna variable residencial adicional"
)

max_missing_feature = feature_info.sort_values(
    "missing_pct",
    ascending=False,
).iloc[0]

display(
    Markdown(
        f"""
### Análisis de las variables

KNN utilizará **{len(KNN_FEATURES)} variables**. Además de precio y tiempo, se han podido
incorporar: **{optional_text}**.

La variable con mayor proporción de valores ausentes es
**`{max_missing_feature['feature']}`**, con un **{max_missing_feature['missing_pct']:.2f} %**.

Los valores ausentes de las variables auxiliares se imputarán con la mediana del destino.
Esta imputación solo se utiliza para construir la distancia de similitud; los objetivos
principales de precio y tiempo no se imputan.
"""
    )
)

## 12.12 Preparación de variables y estandarización

KNN se basa en distancias. Por ello, una variable expresada en miles no debe dominar a otra expresada entre 0 y 1 simplemente por su escala.

Para cada destino:

1. las variables auxiliares ausentes se sustituyen por su mediana;
2. todas las variables se estandarizan con `StandardScaler`;
3. la distancia euclídea se calcula sobre el espacio estandarizado.

No se utiliza separación *train/test* porque KNN no está estimando una variable objetivo: actúa como mecanismo descriptivo de recuperación por similitud.

In [ ]:
def prepare_destination_matrix(group, feature_cols):
    X = group[feature_cols].copy()

    for column in feature_cols:
        X[column] = pd.to_numeric(
            X[column],
            errors="coerce",
        )

        # Imputación local con la mediana del destino.
        median = X[column].median()

        if pd.isna(median):
            median = 0.0

        X[column] = X[column].fillna(median)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X, X_scaled, scaler

## 12.13 Construcción del buscador de vecinos

Se utiliza `NearestNeighbors` con distancia euclídea.

La recomendación de referencia se excluye de sus propios vecinos y el número de resultados se adapta automáticamente si existen menos candidatos que el `k` solicitado.

In [ ]:
DEFAULT_K = 5


def nearest_from_candidate_positions(
    X_scaled,
    anchor_position,
    candidate_positions,
    k,
):
    # Copiamos las posiciones para no modificar el objeto original.
    candidate_positions = list(candidate_positions)

    # La referencia nunca puede ser vecina de sí misma.
    if anchor_position in candidate_positions:
        candidate_positions.remove(anchor_position)

    if len(candidate_positions) == 0:
        return []

    n_neighbors = min(
        k,
        len(candidate_positions),
    )

    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="euclidean",
    )

    candidate_matrix = X_scaled[
        candidate_positions
    ]

    model.fit(candidate_matrix)

    distances, local_indices = model.kneighbors(
        X_scaled[[anchor_position]],
        return_distance=True,
    )

    results = []

    for rank, (distance, local_index) in enumerate(
        zip(distances[0], local_indices[0]),
        start=1,
    ):
        global_position = candidate_positions[
            int(local_index)
        ]

        results.append(
            (
                rank,
                global_position,
                float(distance),
            )
        )

    return results

## 12.14 Comparación KNN global vs Pareto + KNN

Para cada combinación destino–perfil se ejecutan las dos estrategias.

Además de la distancia se guardan:

- pertenencia del vecino a Pareto;
- diferencia porcentual de precio;
- diferencia porcentual de tiempo;
- rango del vecino.

Así se puede evaluar tanto la similitud matemática como el efecto práctico de sustituir la referencia por una alternativa.

In [ ]:
def run_knn_comparison(
    work_df,
    anchors_df,
    feature_cols,
    k=5,
):
    rows = []

    for destination, group_original in work_df.groupby(
        DEST_COL
    ):
        # Reiniciamos el índice para alinear las posiciones con X_scaled.
        group = group_original.reset_index(
            drop=True
        ).copy()

        _, X_scaled, _ = prepare_destination_matrix(
            group,
            feature_cols,
        )

        destination_anchors = anchors_df[
            anchors_df[DEST_COL] == destination
        ]

        for _, anchor_info in destination_anchors.iterrows():
            profile_name = anchor_info["profile"]
            anchor_zone_id = anchor_info["anchor_zone_id"]

            anchor_matches = group.index[
                group[ZONE_ID_COL] == anchor_zone_id
            ].tolist()

            if not anchor_matches:
                continue

            anchor_position = anchor_matches[0]
            anchor_row = group.iloc[
                anchor_position
            ]

            candidate_sets = {
                "KNN_global": group.index.tolist(),
                "Pareto_KNN": group.index[
                    group["is_pareto_knn12"]
                ].tolist(),
            }

            for method, candidate_positions in candidate_sets.items():
                neighbors = nearest_from_candidate_positions(
                    X_scaled=X_scaled,
                    anchor_position=anchor_position,
                    candidate_positions=candidate_positions,
                    k=k,
                )

                for rank, neighbor_position, distance in neighbors:
                    neighbor = group.iloc[
                        neighbor_position
                    ]

                    price_delta_pct = (
                        100
                        * (
                            neighbor[PRICE_COL]
                            - anchor_row[PRICE_COL]
                        )
                        / anchor_row[PRICE_COL]
                    )

                    if np.isclose(
                        anchor_row[TIME_COL],
                        0,
                    ):
                        time_delta_pct = np.nan
                    else:
                        time_delta_pct = (
                            100
                            * (
                                neighbor[TIME_COL]
                                - anchor_row[TIME_COL]
                            )
                            / anchor_row[TIME_COL]
                        )

                    result = {
                        DEST_COL: destination,
                        "profile": profile_name,
                        "method": method,
                        "k": k,
                        "rank": rank,
                        "anchor_zone_id": anchor_zone_id,
                        "neighbor_zone_id": neighbor[ZONE_ID_COL],
                        "knn_distance": distance,
                        "neighbor_is_pareto": bool(
                            neighbor["is_pareto_knn12"]
                        ),
                        "anchor_price_m2": anchor_row[PRICE_COL],
                        "neighbor_price_m2": neighbor[PRICE_COL],
                        "price_delta_pct": price_delta_pct,
                        "anchor_commute_minutes": anchor_row[TIME_COL],
                        "neighbor_commute_minutes": neighbor[TIME_COL],
                        "time_delta_pct": time_delta_pct,
                    }

                    if ZONE_NAME_COL is not None:
                        result["anchor_zone_name"] = anchor_row[
                            ZONE_NAME_COL
                        ]
                        result["neighbor_zone_name"] = neighbor[
                            ZONE_NAME_COL
                        ]

                    rows.append(result)

    return pd.DataFrame(rows)


neighbors_default = run_knn_comparison(
    work_df=work,
    anchors_df=anchors,
    feature_cols=KNN_FEATURES,
    k=DEFAULT_K,
)

if neighbors_default.empty:
    raise ValueError(
        "KNN no ha generado ninguna alternativa."
    )

print(
    f"Alternativas generadas con k={DEFAULT_K}: "
    f"{len(neighbors_default):,}"
)

display(neighbors_default.head(10))

## 12.15 Ejemplo de recomendaciones

La columna `knn_distance` representa la distancia entre cada vecino y la recomendación de referencia en el espacio estandarizado:

- valores pequeños → mayor similitud;
- valores grandes → menor similitud.

Las variaciones porcentuales permiten traducir esa distancia a cambios comprensibles en precio y tiempo.

In [ ]:
example_columns = [
    DEST_COL,
    "profile",
    "method",
    "rank",
    "knn_distance",
    "neighbor_is_pareto",
    "price_delta_pct",
    "time_delta_pct",
]

if "neighbor_zone_name" in neighbors_default.columns:
    example_columns.insert(
        4,
        "neighbor_zone_name",
    )
else:
    example_columns.insert(
        4,
        "neighbor_zone_id",
    )

display(
    neighbors_default[
        example_columns
    ]
    .sort_values(
        [
            DEST_COL,
            "profile",
            "method",
            "rank",
        ]
    )
    .head(40)
)

## 12.16 Métricas de evaluación

Se utilizan cuatro medidas principales:

- **Proporción Pareto:** porcentaje de vecinos no dominados.
- **Distancia KNN media:** similitud respecto a la referencia.
- **Cambio absoluto de precio:** diferencia porcentual media ignorando el signo.
- **Cambio absoluto de tiempo:** diferencia porcentual media ignorando el signo.

El enfoque híbrido no intenta minimizar la distancia a cualquier coste. Busca conservar una similitud razonable garantizando eficiencia en el problema precio–tiempo.

In [ ]:
def evaluate_neighbor_results(
    neighbors_df
):
    if neighbors_df.empty:
        return pd.DataFrame()

    evaluation = (
        neighbors_df
        .assign(
            abs_price_delta_pct=lambda df: (
                df["price_delta_pct"].abs()
            ),
            abs_time_delta_pct=lambda df: (
                df["time_delta_pct"].abs()
            ),
        )
        .groupby(
            [
                DEST_COL,
                "profile",
                "method",
                "k",
            ],
            as_index=False,
        )
        .agg(
            n_alternatives=(
                "neighbor_zone_id",
                "size",
            ),
            pareto_rate=(
                "neighbor_is_pareto",
                "mean",
            ),
            mean_knn_distance=(
                "knn_distance",
                "mean",
            ),
            median_knn_distance=(
                "knn_distance",
                "median",
            ),
            mean_abs_price_change_pct=(
                "abs_price_delta_pct",
                "mean",
            ),
            mean_abs_time_change_pct=(
                "abs_time_delta_pct",
                "mean",
            ),
        )
    )

    evaluation["pareto_rate_pct"] = (
        100 * evaluation["pareto_rate"]
    )

    return evaluation


evaluation_default = evaluate_neighbor_results(
    neighbors_default
)

display(
    evaluation_default.sort_values(
        [
            DEST_COL,
            "profile",
            "method",
        ]
    )
)

In [ ]:
overall_default = (
    evaluation_default
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        mean_pareto_rate_pct=(
            "pareto_rate_pct",
            "mean",
        ),
        mean_knn_distance=(
            "mean_knn_distance",
            "mean",
        ),
        mean_abs_price_change_pct=(
            "mean_abs_price_change_pct",
            "mean",
        ),
        mean_abs_time_change_pct=(
            "mean_abs_time_change_pct",
            "mean",
        ),
    )
)

display(overall_default)

In [ ]:
overall_indexed = overall_default.set_index(
    "method"
)

global_row = overall_indexed.loc[
    "KNN_global"
]
hybrid_row = overall_indexed.loc[
    "Pareto_KNN"
]

pareto_gain = (
    hybrid_row["mean_pareto_rate_pct"]
    - global_row["mean_pareto_rate_pct"]
)

if np.isclose(
    global_row["mean_knn_distance"],
    0,
):
    distance_change_pct = np.nan
else:
    distance_change_pct = (
        100
        * (
            hybrid_row["mean_knn_distance"]
            - global_row["mean_knn_distance"]
        )
        / global_row["mean_knn_distance"]
    )

distance_text = (
    f"{distance_change_pct:.2f} %"
    if pd.notna(distance_change_pct)
    else "no calculable"
)

display(
    Markdown(
        f"""
### Análisis de la comparación principal (`k = {DEFAULT_K}`)

El KNN global obtiene una proporción media de alternativas Pareto del
**{global_row['mean_pareto_rate_pct']:.2f} %**, mientras que **Pareto + KNN**
alcanza **{hybrid_row['mean_pareto_rate_pct']:.2f} %**.

Esto supone una mejora de **{pareto_gain:.2f} puntos porcentuales** en eficiencia.

La distancia media respecto a la referencia pasa de
**{global_row['mean_knn_distance']:.3f}** a
**{hybrid_row['mean_knn_distance']:.3f}**, una variación del **{distance_text}**.

El resultado permite medir el coste real de imponer eficiencia multiobjetivo.
Si la proporción Pareto mejora claramente y el aumento de distancia es moderado,
la combinación Pareto + KNN es más coherente con el objetivo del recomendador
que utilizar KNN sin restricciones.
"""
    )
)

## 12.17 Solapamiento entre estrategias

La distancia media no indica si ambas estrategias devuelven los mismos barrios.

Se utiliza el **índice de Jaccard**:

- `1` → listas idénticas;
- `0` → no comparten ningún barrio.

Así se cuantifica cuánto cambia realmente la composición de las recomendaciones al introducir el filtro Pareto.

In [ ]:
strategy_overlap_rows = []

for (
    destination,
    profile,
), group in neighbors_default.groupby(
    [DEST_COL, "profile"]
):
    global_set = set(
        group.loc[
            group["method"] == "KNN_global",
            "neighbor_zone_id",
        ]
    )

    hybrid_set = set(
        group.loc[
            group["method"] == "Pareto_KNN",
            "neighbor_zone_id",
        ]
    )

    union = global_set | hybrid_set
    intersection = global_set & hybrid_set

    jaccard = (
        len(intersection) / len(union)
        if union
        else np.nan
    )

    strategy_overlap_rows.append(
        {
            DEST_COL: destination,
            "profile": profile,
            "global_neighbors": len(
                global_set
            ),
            "hybrid_neighbors": len(
                hybrid_set
            ),
            "intersection": len(
                intersection
            ),
            "union": len(union),
            "jaccard": jaccard,
        }
    )

strategy_overlap = pd.DataFrame(
    strategy_overlap_rows
)

display(strategy_overlap)

In [ ]:
mean_strategy_jaccard = (
    strategy_overlap["jaccard"].mean()
)

display(
    Markdown(
        f"""
### Análisis del solapamiento entre estrategias

El índice de Jaccard medio entre KNN global y Pareto + KNN es
**{mean_strategy_jaccard:.3f}**.

Cuanto más cercano sea a 1, menos cambia la lista al aplicar Pareto.
Valores inferiores indican que la restricción multiobjetivo sustituye vecinos
globalmente similares por alternativas que cumplen mejor el criterio de
eficiencia precio–tiempo.
"""
    )
)

## 12.18 Sensibilidad al número de vecinos

Para evitar que las conclusiones dependan únicamente de `k = 5`, se repite el análisis con:

- `k = 3`;
- `k = 5`;
- `k = 7`;
- `k = 10`.

Esto permite comprobar si la ventaja del enfoque híbrido se mantiene con listas de diferente longitud.

In [ ]:
K_VALUES = [3, 5, 7, 10]

sensitivity_neighbors_list = []
sensitivity_evaluations_list = []

for k_value in K_VALUES:
    current_neighbors = run_knn_comparison(
        work_df=work,
        anchors_df=anchors,
        feature_cols=KNN_FEATURES,
        k=k_value,
    )

    current_evaluation = evaluate_neighbor_results(
        current_neighbors
    )

    sensitivity_neighbors_list.append(
        current_neighbors
    )
    sensitivity_evaluations_list.append(
        current_evaluation
    )

neighbors_sensitivity = pd.concat(
    sensitivity_neighbors_list,
    ignore_index=True,
)

evaluation_sensitivity = pd.concat(
    sensitivity_evaluations_list,
    ignore_index=True,
)

sensitivity_summary = (
    evaluation_sensitivity
    .groupby(
        ["method", "k"],
        as_index=False,
    )
    .agg(
        mean_pareto_rate_pct=(
            "pareto_rate_pct",
            "mean",
        ),
        mean_knn_distance=(
            "mean_knn_distance",
            "mean",
        ),
        mean_abs_price_change_pct=(
            "mean_abs_price_change_pct",
            "mean",
        ),
        mean_abs_time_change_pct=(
            "mean_abs_time_change_pct",
            "mean",
        ),
    )
)

display(sensitivity_summary)

In [ ]:
hybrid_sensitivity = sensitivity_summary[
    sensitivity_summary["method"]
    == "Pareto_KNN"
].copy()

display(
    Markdown(
        f"""
### Análisis de sensibilidad de `k`

Para Pareto + KNN, la proporción media de alternativas eficientes se mueve entre
**{hybrid_sensitivity['mean_pareto_rate_pct'].min():.2f} %** y
**{hybrid_sensitivity['mean_pareto_rate_pct'].max():.2f} %**.

La distancia media se sitúa entre
**{hybrid_sensitivity['mean_knn_distance'].min():.3f}** y
**{hybrid_sensitivity['mean_knn_distance'].max():.3f}**.

Si la ventaja frente al KNN global se mantiene para todos los valores de `k`,
la conclusión no depende de una configuración puntual del número de vecinos.
"""
    )
)

## 12.19 Visualización: eficiencia Pareto según `k`

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for method, group in sensitivity_summary.groupby(
    "method"
):
    ordered = group.sort_values(
        "k"
    )

    ax.plot(
        ordered["k"],
        ordered["mean_pareto_rate_pct"],
        marker="o",
        label=method,
    )

ax.set_title(
    "Proporción media de alternativas Pareto según k"
)
ax.set_xlabel(
    "Número de vecinos (k)"
)
ax.set_ylabel(
    "Alternativas Pareto (%)"
)
ax.set_xticks(K_VALUES)
ax.grid(alpha=0.25)
ax.legend()

plt.show()

### Lectura del gráfico

La línea de Pareto + KNN debería permanecer en el máximo posible porque sus candidatos ya están restringidos a la frontera eficiente.

La diferencia respecto al KNN global representa cuántas alternativas dominadas podrían introducirse si se utilizara similitud sin el filtro multiobjetivo.

## 12.20 Visualización: coste en similitud según `k`

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for method, group in sensitivity_summary.groupby(
    "method"
):
    ordered = group.sort_values(
        "k"
    )

    ax.plot(
        ordered["k"],
        ordered["mean_knn_distance"],
        marker="o",
        label=method,
    )

ax.set_title(
    "Distancia KNN media según k"
)
ax.set_xlabel(
    "Número de vecinos (k)"
)
ax.set_ylabel(
    "Distancia media estandarizada"
)
ax.set_xticks(K_VALUES)
ax.grid(alpha=0.25)
ax.legend()

plt.show()

### Lectura del gráfico

Es normal que Pareto + KNN tenga una distancia igual o superior a la del KNN global, ya que busca dentro de un conjunto más reducido.

El objetivo no es eliminar totalmente ese incremento, sino comprobar si el coste de similitud es razonable frente al beneficio de garantizar alternativas no dominadas.

## 12.21 Solapamiento entre perfiles

También se comprueba que estudiante, equilibrado y profesional no produzcan siempre las mismas alternativas.

Se calcula Jaccard entre cada par de perfiles usando Pareto + KNN con `k = 5`.

In [ ]:
hybrid_default = neighbors_default[
    neighbors_default["method"]
    == "Pareto_KNN"
].copy()

profile_overlap_rows = []

for (
    destination,
    destination_group,
) in hybrid_default.groupby(
    DEST_COL
):
    profile_sets = {
        profile: set(
            profile_group[
                "neighbor_zone_id"
            ]
        )
        for (
            profile,
            profile_group,
        ) in destination_group.groupby(
            "profile"
        )
    }

    for (
        profile_a,
        profile_b,
    ) in combinations(
        sorted(profile_sets),
        2,
    ):
        set_a = profile_sets[
            profile_a
        ]
        set_b = profile_sets[
            profile_b
        ]

        union = set_a | set_b
        intersection = set_a & set_b

        jaccard = (
            len(intersection)
            / len(union)
            if union
            else np.nan
        )

        profile_overlap_rows.append(
            {
                DEST_COL: destination,
                "profile_a": profile_a,
                "profile_b": profile_b,
                "intersection": len(
                    intersection
                ),
                "union": len(union),
                "jaccard": jaccard,
            }
        )

profile_overlap = pd.DataFrame(
    profile_overlap_rows
)

display(profile_overlap)

In [ ]:
if not profile_overlap.empty:
    mean_profile_jaccard = (
        profile_overlap["jaccard"].mean()
    )

    pair_summary = (
        profile_overlap
        .groupby(
            ["profile_a", "profile_b"],
            as_index=False,
        )
        .agg(
            mean_jaccard=(
                "jaccard",
                "mean",
            )
        )
        .sort_values(
            "mean_jaccard"
        )
    )

    display(pair_summary)

    most_different = pair_summary.iloc[
        0
    ]
    most_similar = pair_summary.iloc[
        -1
    ]

    display(
        Markdown(
            f"""
### Análisis del efecto de los perfiles

El solapamiento medio entre perfiles dentro de Pareto + KNN es
**{mean_profile_jaccard:.3f}**.

El par con menor solapamiento es
**{most_different['profile_a']} – {most_different['profile_b']}**
(`Jaccard = {most_different['mean_jaccard']:.3f}`).

El par con mayor solapamiento es
**{most_similar['profile_a']} – {most_similar['profile_b']}**
(`Jaccard = {most_similar['mean_jaccard']:.3f}`).

Un valor inferior a 1 confirma que las prioridades del usuario modifican,
al menos parcialmente, las alternativas propuestas.
"""
        )
    )
else:
    mean_profile_jaccard = np.nan

    display(
        Markdown(
            "### Análisis del efecto de los perfiles\\n\\n"
            "No hay suficientes perfiles comparables para calcular el solapamiento."
        )
    )

## 12.22 Visualización de un caso concreto

Se representa el primer destino disponible para el perfil equilibrado.

El gráfico muestra:

- todas las alternativas válidas;
- los barrios Pareto;
- la recomendación de referencia;
- los vecinos Pareto + KNN.

Esta visualización ayuda a interpretar la relación entre similitud y eficiencia.

In [ ]:
example_destination = sorted(
    work[DEST_COL]
    .dropna()
    .unique()
)[0]

example_profile = "equilibrado"

plot_group = work[
    work[DEST_COL]
    == example_destination
].copy()

anchor_info = anchors[
    (
        anchors[DEST_COL]
        == example_destination
    )
    & (
        anchors["profile"]
        == example_profile
    )
].iloc[0]

example_neighbors = hybrid_default[
    (
        hybrid_default[DEST_COL]
        == example_destination
    )
    & (
        hybrid_default["profile"]
        == example_profile
    )
]

fig, ax = plt.subplots(
    figsize=(9, 6)
)

# Todas las alternativas comparables.
ax.scatter(
    plot_group[PRICE_COL],
    plot_group[TIME_COL],
    alpha=0.45,
    label="Alternativas",
)

# Alternativas Pareto.
pareto_plot = plot_group[
    plot_group["is_pareto_knn12"]
].sort_values(
    PRICE_COL
)

ax.scatter(
    pareto_plot[PRICE_COL],
    pareto_plot[TIME_COL],
    s=65,
    label="Pareto",
)

# Recomendación de referencia.
ax.scatter(
    [
        anchor_info[
            "anchor_price_m2"
        ]
    ],
    [
        anchor_info[
            "anchor_commute_minutes"
        ]
    ],
    marker="*",
    s=220,
    label="Referencia equilibrada",
)

# Vecinos recuperados por el método híbrido.
ax.scatter(
    example_neighbors[
        "neighbor_price_m2"
    ],
    example_neighbors[
        "neighbor_commute_minutes"
    ],
    marker="x",
    s=90,
    label="Vecinos Pareto + KNN",
)

ax.set_title(
    f"Ejemplo Pareto + KNN · {example_destination}"
)
ax.set_xlabel(
    "Precio de vivienda (€/m²)"
)
ax.set_ylabel(
    "Tiempo diario de desplazamiento (min)"
)
ax.grid(alpha=0.25)
ax.legend()

plt.show()

### Interpretación del caso

La referencia parte de una alternativa eficiente y KNN busca zonas próximas en el espacio de características.

En el enfoque híbrido todos los vecinos deben seguir perteneciendo a la frontera de Pareto. Por tanto, un barrio puede quedar fuera aunque sea muy parecido si existe otra alternativa que lo mejora simultáneamente en precio y tiempo.

La similitud se aplica **después** del criterio multiobjetivo, no al revés.

## 12.23 Comprobaciones finales

Antes de exportar se verifica que:

- todas las referencias sean Pareto;
- todos los vecinos de Pareto + KNN sean Pareto;
- ninguna referencia aparezca como vecina de sí misma;
- las distancias sean no negativas;
- los rangos comiencen en 1;
- no existan vecinos duplicados dentro de una misma lista.

In [ ]:
assert anchors[
    "anchor_is_pareto"
].all()

hybrid_rows = neighbors_default[
    neighbors_default["method"]
    == "Pareto_KNN"
]

assert hybrid_rows[
    "neighbor_is_pareto"
].all(), (
    "Pareto + KNN ha devuelto "
    "una alternativa dominada."
)

assert not (
    neighbors_default[
        "anchor_zone_id"
    ]
    == neighbors_default[
        "neighbor_zone_id"
    ]
).any(), (
    "Una referencia aparece "
    "como vecina de sí misma."
)

assert (
    neighbors_default[
        "knn_distance"
    ] >= 0
).all()

assert (
    neighbors_default[
        "rank"
    ].min() == 1
)

duplicate_neighbor_count = (
    neighbors_default
    .duplicated(
        subset=[
            DEST_COL,
            "profile",
            "method",
            "k",
            "neighbor_zone_id",
        ]
    )
    .sum()
)

assert (
    duplicate_neighbor_count
    == 0
), (
    "Hay vecinos repetidos "
    "dentro de una misma lista."
)

print(
    "Todas las validaciones del notebook 12 "
    "se han superado correctamente."
)

## 12.24 Evaluación de las hipótesis

Las hipótesis no se aceptan mediante un umbral estadístico arbitrario, ya que este notebook no plantea un experimento supervisado.

Se evalúan empíricamente usando:

- distancia de similitud;
- variación de precio y tiempo;
- proporción Pareto;
- sensibilidad a `k`;
- solapamiento entre estrategias;
- solapamiento entre perfiles.

In [ ]:
overall = overall_default.set_index(
    "method"
)

global_row = overall.loc[
    "KNN_global"
]
hybrid_row = overall.loc[
    "Pareto_KNN"
]

pareto_gain_pp = (
    hybrid_row["mean_pareto_rate_pct"]
    - global_row["mean_pareto_rate_pct"]
)

if np.isclose(
    global_row["mean_knn_distance"],
    0,
):
    similarity_cost_pct = np.nan
else:
    similarity_cost_pct = (
        100
        * (
            hybrid_row["mean_knn_distance"]
            - global_row["mean_knn_distance"]
        )
        / global_row["mean_knn_distance"]
    )

strategy_jaccard = (
    strategy_overlap[
        "jaccard"
    ].mean()
    if not strategy_overlap.empty
    else np.nan
)

profile_jaccard = (
    profile_overlap[
        "jaccard"
    ].mean()
    if not profile_overlap.empty
    else np.nan
)

final_summary = pd.DataFrame(
    {
        "metric": [
            "KNN global - Pareto (%)",
            "Pareto + KNN - Pareto (%)",
            "Ganancia Pareto (p.p.)",
            "KNN global - distancia media",
            "Pareto + KNN - distancia media",
            "Coste relativo de similitud (%)",
            "Jaccard entre estrategias",
            "Jaccard medio entre perfiles",
        ],
        "value": [
            global_row[
                "mean_pareto_rate_pct"
            ],
            hybrid_row[
                "mean_pareto_rate_pct"
            ],
            pareto_gain_pp,
            global_row[
                "mean_knn_distance"
            ],
            hybrid_row[
                "mean_knn_distance"
            ],
            similarity_cost_pct,
            strategy_jaccard,
            profile_jaccard,
        ],
    }
)

display(final_summary)

In [ ]:
similarity_cost_text = (
    f"{similarity_cost_pct:.2f} %"
    if pd.notna(
        similarity_cost_pct
    )
    else "no calculable"
)

display(
    Markdown(
        f"""
### Resultado general

El KNN aplicado sin filtro multiobjetivo obtiene una proporción media de alternativas
Pareto del **{global_row['mean_pareto_rate_pct']:.2f} %**. Al restringir la búsqueda
a la frontera de Pareto, esta proporción pasa al
**{hybrid_row['mean_pareto_rate_pct']:.2f} %**.

La mejora es de **{pareto_gain_pp:.2f} puntos porcentuales**.

La distancia media respecto a la recomendación de referencia es
**{global_row['mean_knn_distance']:.3f}** con KNN global y
**{hybrid_row['mean_knn_distance']:.3f}** con Pareto + KNN.
El coste relativo de similitud es **{similarity_cost_text}**.

El solapamiento medio entre ambas estrategias es
**{strategy_jaccard:.3f}** según Jaccard. El solapamiento medio entre perfiles
dentro del enfoque híbrido es **{profile_jaccard:.3f}**.

Estos resultados permiten valorar directamente el compromiso entre **similitud**
y **eficiencia multiobjetivo**. Pareto sigue decidiendo qué alternativas son
eficientes y KNN añade una capa de similitud sobre esas soluciones.
"""
    )
)

## 12.25 Exportación de resultados

Se guardan las recomendaciones y las tablas de evaluación para reutilizarlas en la memoria del TFG o en una futura interfaz.

In [ ]:
OUTPUTS = {
    "knn12_anchors.csv": anchors,
    "knn12_feature_info.csv": feature_info,
    "knn12_neighbors_k5.csv": neighbors_default,
    "knn12_evaluation_k5.csv": evaluation_default,
    "knn12_strategy_overlap.csv": strategy_overlap,
    "knn12_neighbors_sensitivity.csv": neighbors_sensitivity,
    "knn12_sensitivity_summary.csv": sensitivity_summary,
    "knn12_profile_overlap.csv": profile_overlap,
    "knn12_final_summary.csv": final_summary,
}

for filename, dataframe in OUTPUTS.items():
    output_path = (
        RESULTS_DIR / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print(
        f"Guardado: {output_path}"
    )

## 12.26 Conclusiones

Este notebook incorpora KNN de forma complementaria al recomendador desarrollado en los notebooks anteriores.

La principal diferencia respecto a un sistema basado únicamente en vecinos cercanos es que el enfoque híbrido **mantiene primero la condición de eficiencia Pareto**. Así se evita recomendar una zona solo porque sea parecida a la referencia cuando existe otra alternativa que la mejora simultáneamente en precio y tiempo.

KNN aporta una utilidad distinta: permite pasar de una única recomendación a un pequeño conjunto de **alternativas similares**, especialmente útil cuando el usuario quiere comparar varias zonas con características próximas.

La comparación con KNN global cuantifica el coste de imponer la restricción multiobjetivo. Si la distancia aumenta de forma limitada mientras la proporción de alternativas Pareto mejora claramente, existe evidencia a favor de combinar ambos métodos.

También se analiza la sensibilidad al número de vecinos y el solapamiento entre perfiles, evitando que las conclusiones dependan de una única configuración de `k` y comprobando que las preferencias continúan influyendo en las recomendaciones.

En conjunto, **Pareto sigue siendo el núcleo del sistema y KNN funciona como una capa auxiliar de similitud**. Esta separación mantiene el foco del TFG en el equilibrio entre coste de vivienda y accesibilidad, pero añade una técnica de aprendizaje automático con una función práctica, interpretable y evaluable.

## 12.27 Limitaciones y trabajo futuro

La noción de similitud de KNN depende de las variables disponibles. En el futuro podrían incorporarse variables urbanas como servicios, zonas verdes, oferta educativa o equipamientos.

La distancia KNN tampoco representa directamente satisfacción del usuario. Dos barrios próximos en el espacio de características pueden presentar diferencias cualitativas no recogidas por el dataset.

Además, el análisis trabaja con valores agregados a nivel de barrio, por lo que las recomendaciones describen zonas y no viviendas concretas.

Finalmente, Pareto se construye con dos objetivos principales. Si se añadieran nuevos objetivos independientes sería necesario estudiar cómo cambia la frontera y la utilidad de KNN en un espacio de decisión más amplio.

## 12.28 Resultado del notebook

El resultado principal es una extensión híbrida que combina:

- **optimización multiobjetivo**, mediante Pareto;
- **preferencias del usuario**, mediante perfiles ponderados;
- **similitud entre barrios**, mediante KNN;
- **evaluación cuantitativa**, mediante eficiencia, distancia, variación y solapamiento;
- **análisis de sensibilidad**, mediante distintos valores de `k`.

Los archivos `knn12_neighbors_k5.csv`, `knn12_sensitivity_summary.csv` y `knn12_final_summary.csv` recogen los resultados principales del experimento.